# ParaSpeechCaps

## EARS

In [ ]:
import json
import pandas as pd


with open("C:/Users/jackm/Documents/GitHub/ms-capstone-TTS-natlang-styleprompts/src/data/raw/paraspeechcaps/audio/EARS/speaker_statistics.json", "r") as f:
    data = json.load(f)

# keys -> index, then move index into a column
df = pd.DataFrame.from_dict(data, orient="index").reset_index()

# rename the index column
df = df.rename(columns={"index": "speaker_id"})

df

Show distributions of stats in EARS:

In [ ]:
import matplotlib.pyplot as plt

stat_columns = [c for c in df.columns if c != "speaker_id"]

for col in stat_columns:
    plt.figure()
    ax = df[col].value_counts().sort_index().plot(kind="bar")

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("count")

    # add numbers on top of bars
    for p in ax.patches:
        ax.annotate(
            str(int(p.get_height())),
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()


To get a more balanced speaker set, we want to reduce some speakers of the majority classes (white/caucasian, american english).
This information will help us choose which files to not download from EARS.
Hopefully this allows the model to generalize more.

In [18]:
criteria = {
    "ethnicity": "white or caucasian",
    "native language" : "american english"
}

n = 20  # number of rows to pick

# Step 1: create mask for rows matching all criteria
mask = (df[list(criteria)] == pd.Series(criteria)).all(axis=1)

# Step 2: get the matching rows
matching = df[mask]

# Step 3: sample n rows from matches (safe if fewer than n exist)
selected = matching.sample(n=min(n, len(matching)), random_state=42)

# Step 4: remove selected rows from original dataframe
df_balanced = df.drop(selected.index)

df_balanced

,speaker_id,age,ethnicity,gender,weight,native language,height
0,p001,36-45,white or caucasian,male,160 - 180 lbs,german,6' - 6'3
2,p003,26-35,black or african american,female,100 - 120 lbs,american english,5'4 - 5'7
3,p004,18-25,white or caucasian,male,140 - 160 lbs,american english,5'8 - 5'11
4,p005,18-25,asian,male,140 - 160 lbs,mandarin,5'4 - 5'7
5,p006,36-45,hispanic or latino,female,200 - 220 lbs,american english,< 5'
...,...,...,...,...,...,...,...
102,p103,prefer not to answer,prefer not to answer,prefer not to answer,prefer not to answer,prefer not to answer,prefer not to answer
103,p104,56-65,white or caucasian,female,180 - 200 lbs,american english,5'4 - 5'7
104,p105,36-45,black or african american,male,220 - 240 lbs,american english,6' - 6'3
105,p106,56-65,white or caucasian,female,120 - 140 lbs,american english,5' - 5'3


new distributions:

In [ ]:
import matplotlib.pyplot as plt

stat_columns = [c for c in df_balanced.columns if c != "speaker_id"]

for col in stat_columns:
    plt.figure()
    ax = df_balanced[col].value_counts().sort_index().plot(kind="bar")

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("count")

    # add numbers on top of bars
    for p in ax.patches:
        ax.annotate(
            str(int(p.get_height())),
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

In [22]:
# get speaker ids to EXCLUDE
list(selected["speaker_id"])

['p034',
 'p002',
 'p076',
 'p011',
 'p084',
 'p028',
 'p019',
 'p051',
 'p066',
 'p022',
 'p049',
 'p018',
 'p102',
 'p014',
 'p093',
 'p048',
 'p088',
 'p054',
 'p087',
 'p070']